# Example 0.13.3 / 6: Exercise 20-1: Choosing Advertising Media

Companion notebook for **0.13.3 Exercises on Special Structures** in the AMPL lab Documentation, section "Exercise 20-1: Choosing Advertising Media". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 20-1: Choosing Advertising Media

Exercise 20-1 allocates a \$1 million campaign when each medium is either used fully or not at all.
Each medium has an audience and a cost, in thousands, and uses creative time of three kinds:

| | TV | Magazine | Radio | Newspaper | Mail | Phone | Limit |
|---|---|---|---|---|---|---|---|
| Audience | 300 | 200 | 100 | 150 | 100 | 50 | |
| Cost | 600 | 250 | 100 | 120 | 200 | 250 | 1000 |
| Writers | 120 | 50 | 20 | 40 | 30 | 5 | 200 |
| Artists | 120 | 80 | 0 | 60 | 40 | 0 | 300 |
| Others | 20 | 20 | 20 | 20 | 20 | 200 | 200 |

One binary variable per medium makes this a multidimensional knapsack problem
(Knapsack, Covering and Packing (AMPL lab documentation, section 7.11)). The exercise also asks for the LP relaxation, which
`relax_integrality` solves without changing the model (Relaxations, Bounds and Optimality Gaps (AMPL lab documentation, section 7.1.1)).

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set MEDIA;                                  # advertising media
set RES;                                    # cost and creative resources
param audience {MEDIA} >= 0;                # thousands of people reached
param use {RES, MEDIA} >= 0;                # resource used if a medium is chosen
param limit {RES} >= 0;                     # resource available

var Choose {MEDIA} binary;                  # 1 if the medium is used

maximize Total_Audience: sum {m in MEDIA} audience[m] * Choose[m];  # thousands reached

subject to Limit {r in RES}: sum {m in MEDIA} use[r,m] * Choose[m] <= limit[r];  # within each limit

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

media = ["TV", "Magazine", "Radio", "Newspaper", "Mail", "Phone"]
use = pd.DataFrame(
    [[600, 250, 100, 120, 200, 250], [120, 50, 20, 40, 30, 5],
     [120, 80, 0, 60, 40, 0], [20, 20, 20, 20, 20, 200]],
    index=["Cost", "Writers", "Artists", "Others"], columns=media,
)
ampl.set["MEDIA"] = media
ampl.set["RES"] = list(use.index)
ampl.param["audience"] = dict(zip(media, [300, 200, 100, 150, 100, 50]))
ampl.param["use"] = use
ampl.param["limit"] = {"Cost": 1000, "Writers": 200, "Artists": 300, "Others": 200}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["Total_Audience"].value(),
      [m for m, v in ampl.var["Choose"].to_dict().items() if v > 0.5])
ampl.option["relax_integrality"] = 1
ampl.solve(solver="highs")
print("relaxed:", round(ampl.obj["Total_Audience"].value(), 2),
      {m: round(v, 3) for m, v in ampl.var["Choose"].to_dict().items() if v > 1e-6})

HiGHS 1.14.0: optimal solution; objective 600
5 simplex iterations
1 branching nodes
solved 600.0 ['Magazine', 'Radio', 'TV']
HiGHS 1.14.0: optimal solution; objective 705
2 simplex iterations
0 barrier iterations
relaxed: 705.0 {'Magazine': 1, 'Mail': 1, 'Newspaper': 1, 'Phone': 0.133, 'Radio': 1, 'TV': 0.494}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set MEDIA := TV Magazine Radio Newspaper Mail Phone;
set RES := Cost Writers Artists Others;
param audience := TV 300  Magazine 200  Radio 100  Newspaper 150  Mail 100  Phone 50;
param use:  TV  Magazine  Radio  Newspaper  Mail  Phone :=
Cost       600     250     100      120      200    250
Writers    120      50      20       40       30      5
Artists    120      80       0       60       40      0
Others      20      20      20       20       20    200;
param limit := Cost 1000  Writers 200  Artists 300  Others 200;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Total_Audience, Choose;
option relax_integrality 1;
solve;
display Total_Audience, Choose;

HiGHS 1.14.0: optimal solution; objective 600
5 simplex iterations
1 branching nodes
Total_Audience = 600

Choose [*] :=
 Magazine  1
     Mail  0
Newspaper  0
    Phone  0
    Radio  1
       TV  1
;

HiGHS 1.14.0: optimal solution; objective 705
2 simplex iterations
0 barrier iterations
Total_Audience = 705

Choose [*] :=
 Magazine  1
     Mail  1
Newspaper  1
    Phone  0.133333
    Radio  1
       TV  0.494444
;

